---
# Chapter 1 — What Remembering Means

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 1: What Remembering Means |
| Central question | What should count as memory in an AI assistant, as distinct from storage and retrieval? |
| Main concepts | Storage, Retrieval, Behavioural memory, Perfect Memory Paradox, Historical record, Selective memory, Epistemic status |
| Implementation | None (foundational chapter) |
| Experiment | Pending (requires Chapter 2 instrument) |
| Evidence status | Foundational hypothesis |
| Depends on | None |

---

## What this notebook demonstrates

This chapter establishes the behavioural definition of memory that governs the entire book. The notebook demonstrates the core distinction through a counterfactual: **hold a present task fixed, vary retained history, and show why merely finding history is not yet behavioural memory.**

Since this is a foundational chapter with no implementation yet, this notebook:

1. **Defines the three concepts** (storage, retrieval, memory) operationally
2. **Illustrates the Perfect Memory Paradox** — complete retention ≠ useful memory
3. **Shows the counterfactual test** that Chapter 2 will make measurable
4. **Introduces the running example** (event-store decision: SQLite → PostgreSQL) that persists through the book

> **Evidence status**: This chapter proposes hypotheses and definitions. No experiment has been run yet. The Measurement Instrument in Chapter 2 will make these testable.

## The chapter question

> **What distinguishes storage, retrieval, and memory?**

The book's central claim: **Memory means the preserved past changes what the system does now, judged counterfactually and by improvement.** Storage preserves the past; retrieval locates it; only improved present behaviour counts as memory.

## Concepts in this chapter

The following concepts are introduced here and tracked through the book's metadata:

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(1)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 1 Concepts")

## The running example

The book uses a recurring trio of situations (from the chapter text):

1. **The event-store decision** — Discussion of SQLite vs PostgreSQL across sessions, ending in recorded decision `adr-007` (11 July) to use PostgreSQL
2. **The rejected cache** — Proposal to introduce Redis, exploratory discussion, then recorded decision *not* to introduce it (`adr-009`)
3. **The superseded fact** — A claim that holds for an interval then stops (e.g., which store is production, who owns a component)

These are the canonical identifiers from the repository's controlled fixtures. Let's load the fixture to see the actual artifacts.

In [ ]:
from notebooks.memory._support import load_fixtures

fixtures = load_fixtures("tasks.json")
print(f"Fixture keys: {list(fixtures.keys())}")
print(f"Number of tasks: {len(fixtures.get('tasks', []))}")

# Show the first few tasks to understand the structure
for task in fixtures.get('tasks', [])[:3]:
    print(f"\nTask: {task.get('task_id')}")
    print(f"  Family: {task.get('family')}")
    print(f"  Prompt: {task.get('prompt')}")
    print(f"  Expected state: {task.get('expected_state')}")
    print(f"  Expected sources: {task.get('expected_sources')}")
    print(f"  Superseded options: {task.get('superseded_options')}")
    print(f"  Temporal mode: {task.get('temporal_mode')}")

## The mechanism: The Behavioural Definition

The book defines memory through a **counterfactual ladder**:

```text
past preserved
    ↓
past retrievable
    ↓
past used
    ↓
behaviour changed
    ↓
behaviour improved
```

A system can pass one rung and fail the next. Let's encode this as a testable definition.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class MemoryRung:
    """One rung of the memory evidence ladder."""
    name: str
    description: str
    test: str  # How to verify this rung

RUNGS = [
    MemoryRung(
        "Storage",
        "The past has been preserved and can in principle be found",
        "Check: does the artifact exist in the corpus?"
    ),
    MemoryRung(
        "Retrieval",
        "Some preserved past can be located on demand given a query",
        "Check: does a query return the relevant artifact?"
    ),
    MemoryRung(
        "Use",
        "The retrieved past is used in producing the answer",
        "Check: does the answer cite or visibly depend on the retrieved past?"
    ),
    MemoryRung(
        "Behaviour changed",
        "The past changes what the system does now (counterfactual)",
        "Check: remove the past, rerun the task — is the output different?"
    ),
    MemoryRung(
        "Behaviour improved",
        "The change better serves current goals and constraints",
        "Check: evaluate both outputs against ground truth — which is better?"
    ),
]

render_table([
    {"Rung": r.name, "Description": r.description, "Test": r.test}
    for r in RUNGS
], "The Memory Evidence Ladder (Chapter 1)")

## Use the real implementation

This chapter has no implementation yet — it establishes the *questions* that the implementation must answer. The first real implementation appears in Chapter 3 (the RAG baseline).

However, we can demonstrate the **conceptual mechanism** using the fixture data. Let's trace through the event-store example:

In [ ]:
# The canonical history from the chapter
canonical_history = [
    {
        "id": "session-031",
        "date": "2024-06-15",
        "type": "session",
        "content": "Team discusses event log backend. Proposal: use SQLite for simplicity."
    },
    {
        "id": "session-033",
        "date": "2024-06-22",
        "type": "session",
        "content": "SQLite prototype works but slows under concurrent writes. Contention observed."
    },
    {
        "id": "session-035",
        "date": "2024-07-01",
        "type": "session",
        "content": "Benchmark: PostgreSQL handles 10x concurrent writes. Incident reported from production."
    },
    {
        "id": "adr-007",
        "date": "2024-07-11",
        "type": "decision_record",
        "content": "DECIDED: New event-store work targets PostgreSQL. SQLite prototype superseded.",
        "supersedes": ["session-031"]
    },
    {
        "id": "session-072",
        "date": "2024-10-15",
        "type": "session",
        "content": "New contributor asks assistant to scaffold second service with event log."
    },
]

render_table(canonical_history, "Canonical Event-Store History (from Chapter 1)")

## Run / inspect the example

Now let's simulate the **counterfactual test** from the chapter:

> **October task**: Scaffold a new service with its own event log.
> **Condition A (no relevant history)**: Assistant has no access to July decision.
> **Condition B (relevant history present)**: Assistant has access to `adr-007` and supporting evidence.

The chapter argues: only if Condition B produces *better* behaviour (scaffolds on PostgreSQL with explanation) does the system demonstrate memory.

In [ ]:
# Simulating the counterfactual (conceptual - no model calls)
def simulate_assistant(history_available: bool) -> dict:
    """Conceptual simulation of the two conditions."""
    if not history_available:
        return {
            "action": "scaffold SQLite event store",
            "explanation": "Using SQLite for simplicity (default choice)",
            "memory_rungs": {
                "storage": False,
                "retrieval": False,
                "use": False,
                "behaviour_changed": False,
                "behaviour_improved": False,
            }
        }
    else:
        return {
            "action": "scaffold PostgreSQL event store",
            "explanation": "Team decided PostgreSQL in adr-007 (Jul 11) after SQLite contention benchmark showed 10x worse concurrent writes",
            "memory_rungs": {
                "storage": True,
                "retrieval": True,
                "use": True,
                "behaviour_changed": True,
                "behaviour_improved": True,  # Avoids known failure
            }
        }

cond_a = simulate_assistant(False)
cond_b = simulate_assistant(True)

print("=== CONDITION A: No relevant history ===")
print(f"Action: {cond_a['action']}")
print(f"Explanation: {cond_a['explanation']}")
print(f"Rungs passed: {sum(cond_a['memory_rungs'].values())}/5")

print("\n=== CONDITION B: Relevant history present ===")
print(f"Action: {cond_b['action']}")
print(f"Explanation: {cond_b['explanation']}")
print(f"Rungs passed: {sum(cond_b['memory_rungs'].values())}/5")

print("\n=== COUNTERFACTUAL RESULT ===")
print(f"Behaviour changed: {cond_a['action'] != cond_b['action']}")
print(f"Behaviour improved: {cond_b['memory_rungs']['behaviour_improved']}")

## What happened?

The counterfactual demonstrates the book's central distinction:

- **Storage alone**: The July history exists in the repository (Condition A has storage but no retrieval)
- **Retrieval alone**: A search tool could return `adr-007` when asked (but the system might still scaffold SQLite)
- **Memory**: The system *uses* the retrieved history to change its behaviour *for the better* (Condition B)

The chapter also introduces the **Perfect Memory Paradox**: a system that retains everything (every conversation, every prompt, every tool invocation) creates selection, currency, and relevance problems rather than solving memory. An infinite archive does not solve the memory problem — it makes it visible.

## Connect this to the experiment

Chapter 2 builds the **Memory Measurement Instrument** to make this counterfactual rigorous and measurable:

- Fixed corpus and queries (no builder selection bias)
- Hidden ground-truth ledger (no circular evaluation)
- Per-dimension scorecard (no single aggregate hiding failures)
- Baseline ladder with adversarial controls
- Frozen run manifests for reproducibility

The instrument is the apparatus; the memory system is the subject. They develop together.

## What this establishes

- **Storage ≠ Retrieval ≠ Memory** — three distinct rungs on the evidence ladder
- **Behavioural definition**: Memory requires counterfactual behaviour change *and* improvement
- **Perfect Memory Paradox**: Complete retention creates new problems (selection, currency, relevance)
- **Historical record vs selective memory**: Fidelity underneath; selective, lossy, constructive above
- **Epistemic status matters**: Memory must preserve *how* something is known (hypothesis, evidence, supersession)

## What this does NOT establish

- No system has been built or measured yet
- No taxonomy of human memory categories is assumed necessary
- The six questions (locate, decide, justify, track, unfinished, select) are a hypothesis ladder, not a commitment
- The instrument (Chapter 2) does not yet exist as a runnable artifact

## Try it yourself

Modify the `simulate_assistant` function above to test a third condition: **what if the history is present but *misleading*?** (e.g., the retrieved `adr-007` says PostgreSQL but the current production system still runs SQLite until July 22).

In [ ]:
# TRY IT YOURSELF: Add a misleading history condition

def simulate_assistant_v2(history_condition: str) -> dict:
    """
    history_condition: 'none' | 'correct' | 'misleading'
    - 'none': no history available
    - 'correct': adr-007 (PostgreSQL decision) available
    - 'misleading': only session-031 (SQLite proposal) available
    """
    if history_condition == 'none':
        return {"action": "scaffold SQLite", "explanation": "Default choice", "improved": False}
    elif history_condition == 'correct':
        return {"action": "scaffold PostgreSQL", "explanation": "Per adr-007 (Jul 11)", "improved": True}
    elif history_condition == 'misleading':
        return {"action": "scaffold SQLite", "explanation": "Session-031 proposed SQLite", "improved": False}
    else:
        raise ValueError(f"Unknown condition: {history_condition}")

for cond in ['none', 'correct', 'misleading']:
    result = simulate_assistant_v2(cond)
    print(f"{cond:12s} → {result['action']:25s} | Improved: {result['improved']} | {result['explanation']}")

## Where this leads next

Chapter 2 builds the **Memory Measurement Instrument** — the fixed apparatus that will let us measure whether any memory system actually remembers. The instrument defines:

- Task families (locate, decide, justify, temporal, unfinished, select)
- Hidden ground-truth ledger (evaluator-only)
- Baseline ladder (no memory → lexical → dense → hybrid → reranked → best → oracle)
- Failure taxonomy (attribution to ingestion, retrieval, reconstruction, provenance, temporal, ...)
- Validity checks (oracle positive control, fluent summariser rejection, metric-behaviour bridge)

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)